1. Выберите 10 самых красивых по вашему мнению пятерок в тренировочной выборке MNIST.
2. Создайте датасет, где объекты — это все пятерки из тренировочной части MNIST, а метки — случайные пятерки из выбранного эталонного набора.
3. Создайте автокодировщик и проверьте, совпадают ли у него размеры выхода и входа.
4. Обучите автокодировщик.
5. Добейтесь ошибки MSE на тренировочной выборке меньше 0.05.
6. Посмотрите, как выглядят пятерки из тестовой выборки после обученного автокодировщика.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import MNIST

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Устройство:", device)


In [ ]:
train_base = MNIST(root="data", train=True, download=True)
test_base = MNIST(root="data", train=False, download=True)

x_train_all = train_base.data.float().unsqueeze(1) / 255.0
y_train_all = train_base.targets

x_test_all = test_base.data.float().unsqueeze(1) / 255.0
y_test_all = test_base.targets

train_mask = y_train_all == 5
test_mask = y_test_all == 5

X_train = x_train_all[train_mask]
X_test = x_test_all[test_mask]

print(f"Размер обучающей выборки пятерок: {tuple(X_train.shape)}")
print(f"Размер тестовой выборки пятерок: {tuple(X_test.shape)}")


In [ ]:
beautiful_indexes = torch.tensor([0, 1, 2, 3, 4, 7, 9, 11, 12, 13])
beautiful_fives = X_train[beautiful_indexes].clone()

plt.figure(figsize=(10, 2))
for number, image in enumerate(beautiful_fives):
    plt.subplot(1, 10, number + 1)
    plt.imshow(image.squeeze(0), cmap="gray")
    plt.axis("off")
plt.suptitle("Выбранные эталонные пятерки")
plt.show()

train_generator = torch.Generator().manual_seed(42)
test_generator = torch.Generator().manual_seed(84)

train_target_ids = torch.randint(0, len(beautiful_fives), (len(X_train),), generator=train_generator)
test_target_ids = torch.randint(0, len(beautiful_fives), (len(X_test),), generator=test_generator)

y_train_beautiful = beautiful_fives[train_target_ids].clone()
y_test_beautiful = beautiful_fives[test_target_ids].clone()

train_loader = DataLoader(
    TensorDataset(X_train, y_train_beautiful),
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    TensorDataset(X_test, y_test_beautiful),
    batch_size=64,
    shuffle=False
)


In [ ]:
class ProAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.Flatten()
        )
        self.encoder_dense = nn.Sequential(
            nn.Linear(7 * 7 * 64, 128),
            nn.ReLU(),
            nn.Linear(128, 32),
            nn.ReLU()
        )
        self.decoder_dense = nn.Sequential(
            nn.Linear(32, 7 * 7 * 64),
            nn.ReLU()
        )
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(64, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 1, kernel_size=3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, images):
        encoded = self.encoder_conv(images)
        encoded = self.encoder_dense(encoded)
        decoded = self.decoder_dense(encoded)
        decoded = decoded.view(-1, 64, 7, 7)
        return self.decoder_conv(decoded)


autoencoder = ProAutoencoder().to(device)

with torch.no_grad():
    check_input = torch.zeros(1, 1, 28, 28).to(device)
    check_output = autoencoder(check_input)

print(f"Совпадение размеров входа и выхода: {tuple(check_input.shape[1:]) == tuple(check_output.shape[1:])}")
print(f"Размер входа: {tuple(check_input.shape[1:])} | Размер выхода: {tuple(check_output.shape[1:])}")


In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=0.001)

history = {
    "loss": [],
    "val_loss": []
}

epochs = 15

print("Старт обучения...")
for epoch in range(epochs):
    autoencoder.train()
    train_loss = 0.0

    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        predicted = autoencoder(batch_x)
        loss = criterion(predicted, batch_y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_x.size(0)

    autoencoder.eval()
    val_loss = 0.0

    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            predicted = autoencoder(batch_x)
            loss = criterion(predicted, batch_y)
            val_loss += loss.item() * batch_x.size(0)

    train_loss /= len(train_loader.dataset)
    val_loss /= len(test_loader.dataset)

    history["loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    print(f"Эпоха {epoch + 1:02d}/{epochs} | MSE обучение: {train_loss:.5f} | MSE проверка: {val_loss:.5f}")

final_loss = history["loss"][-1]

print(f"Итоговая ошибка MSE на обучении: {final_loss:.5f}")
if final_loss < 0.05:
    print("Условие задания выполнено: MSE меньше 0.05.")
else:
    print("Ошибка выше 0.05. Нужно увеличить число эпох или усилить модель.")


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history["loss"], label="обучение")
plt.plot(history["val_loss"], label="проверка")
plt.xlabel("Эпоха")
plt.ylabel("MSE")
plt.title("Ошибка автокодировщика")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
autoencoder.eval()

with torch.no_grad():
    predicted_fives = autoencoder(X_test.to(device)).cpu()

plt.figure(figsize=(10, 4))
for i in range(5):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_test[i].squeeze(0), cmap="gray")
    plt.title("Вход")
    plt.axis("off")

    plt.subplot(2, 5, i + 6)
    plt.imshow(predicted_fives[i].squeeze(0), cmap="gray")
    plt.title("Выход")
    plt.axis("off")

plt.suptitle("Преобразование тестовых пятерок")
plt.tight_layout()
plt.show()
